# Assignment 3 — Image Enhancement in the Spatial Domain

Implements bit-plane slicing and 3x3 mean/median noise filtering.

This notebook is intentionally modular. **Style 1** uses OpenCV operations; **Style 2** uses OpenCV only for reading and Matplotlib for display, while the image-processing logic is written with arrays, loops, queues, and formulas.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()

def show_images(images, titles, cmap="gray", figsize=(14, 4)):
    """Display a sequence of images consistently in a notebook."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    axes = np.atleast_1d(axes)
    for axis, image, title in zip(axes, images, titles):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


## Style 1 — OpenCV built-in operations

In [ ]:
def bit_planes_cv(image):
    return [((image >> bit) & 1) * 255 for bit in range(8)]

def denoise_cv(image):
    return cv2.blur(image, (3, 3)), cv2.medianBlur(image, 3)


## Style 2 — DSA/manual implementation

In [ ]:
def bit_planes_manual(image):
    planes = []
    for bit in range(8):
        plane = np.zeros(image.shape, np.uint8)
        for r in range(image.shape[0]):
            for c in range(image.shape[1]):
                plane[r, c] = 255 if (int(image[r, c]) // (2 ** bit)) % 2 else 0
        planes.append(plane)
    return planes

def window_filter_manual(image, mode):
    padded = np.pad(image, 1, mode="edge"); result = np.zeros_like(image)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            values = padded[r:r + 3, c:c + 3].ravel()
            result[r, c] = sum(values) // 9 if mode == "mean" else sorted(values)[4]
    return result.astype(np.uint8)


## Demonstration

In [ ]:
image = cv2.imread(str(LAB_DIR / "Cameraman.png"), cv2.IMREAD_GRAYSCALE)
if image is None: image = np.tile(np.arange(128, dtype=np.uint8), (128, 1))
image = cv2.resize(image, (128, 128))
planes_cv, planes_manual = bit_planes_cv(image), bit_planes_manual(image)
print("Bit-plane equality:", all(np.array_equal(a, b) for a, b in zip(planes_cv, planes_manual)))
show_images([image, denoise_cv(image)[0], window_filter_manual(image, "median")], ["Original", "cv2 mean", "Manual median"])
